# Compare Experimental Conditions
We use the LATER model [1-2] to analyze saccade latencies from two experiments published in Toledano, Sasi et al. (2024) [3]. For each subject, we split saccade to early/late (based on predetermined threshold $T_{thresh}=230ms$) and plot the reciprobit plot for each of the following experimental conditions:
- Cue validity (`valid` / `invalid`)
- Search difficulty (`easy` / `mixed` / `hard`)
- Search difficulty × Cue Validity interaction (late saccades only)
- Cue size (`small` / `large`) - experiment 2 only

If any of these conditions has and effect across trials (meaning, when the data is pooled across all 4 locations), we would see one of the following changes in the reciprobit lines:
- A `shift` indicates a change in _Evidence Accumulation Rate_ ($\mu$).
- A `swivel` (change in slope) indicates a change in _Decision Threshold_ ($\Delta S$) or _Evidence Accumulation Variability_ ($\sigma$).
<br>

```
CITATIONS

[1] Carpenter, R. H., & Williams, M. L. L. (1995). Neural computation of log likelihood in control of saccadic eye movements. Nature, 377(6544), 59-62.

[2] Noorani, I. (2014). LATER models of neural decision behavior in choice tasks. Frontiers in Integrative Neuroscience, 8, 67.

[3] Toledano, D., Sasi, M., Yuval-Greenberg, S., & Lamy, D. (2024). On the timing of overt attention deployment: Eye-movement evidence for the priority accumulation framework. Journal of Experimental Psychology: Human Perception and Performance, 50(5), 431.
```

In [ ]:
import os
from typing import Literal, List, Dict
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pylater

import enum_types as et
from load_data import load_and_prepare_experiments

#### Plotting Function

In [ ]:
def plot_by_condition(
        exp: Literal["exp_1", "exp_2"],
        condition_columns: List[str],
        color_map: Dict[tuple, str],
        late_only: bool,
        subject_col: str = "subject",
        latency_col: str = "saccade_onset",
        is_early_col: str = "is_early_saccade",
) -> None:
    data = DATA[DATA["experiment"] == exp]
    expected_columns = condition_columns + [subject_col, latency_col]
    if late_only and (is_early_col not in expected_columns):
        expected_columns += [is_early_col]
    missing_columns = [col for col in expected_columns if col not in data.columns]
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")
    if late_only:
        data = data[~(data[is_early_col])]
    unique_levels_per_col = [data[col].unique() for col in condition_columns]
    for subj in data[subject_col].unique():
        subj_df = data[data[subject_col] == subj]
        plot = pylater.ReciprobitPlot()
        for combo in product(*unique_levels_per_col):
            # build mask & label parts
            mask = (subj_df[condition_columns[0]] == combo[0])
            label_parts = [_get_label_for_condition(condition_columns[0], combo[0])]
            for col, val in zip(condition_columns[1:], combo[1:]):
                mask &= (subj_df[col] == val)
                label_parts += [_get_label_for_condition(col, val)]
            # extract latencies and plot
            latencies = subj_df.loc[mask, latency_col].dropna()
            latencies /= 1000.0  # ms → s
            label = " / ".join(label_parts) + f" (N={len(latencies)})"
            ds = pylater.Dataset(name=label, rt_s=latencies)
            color = color_map.get(combo, "gray")
            if color == "gray":
                print(f"Warning: Using default color for condition {combo} in experiment {exp}, subject {subj}")
            plot.plot_data(ds, plot_type="scatter", label=label, c=color)
            # set plot's min/max values
            plot.min_rt_s = min(plot.min_rt_s, latencies.min())
            plot.max_rt_s = max(plot.max_rt_s, latencies.max())
        plot.fig = plot.fig.suptitle(f"Experiment {exp[-1]} - Subject {subj}", y=0.98, x=0.1)
        plt.show()
        print()
    return


def _get_label_for_condition(column: str, value) -> str:
    if "is_early" in column:
        if isinstance(value, (bool, np.bool_)):
            return "Early" if value else "Late"
        raise TypeError(f"Unexpected type for column {column}: {type(value)}")
    if "is_valid" in column:
        if isinstance(value, (bool, np.bool_)):
            return "Valid" if value else "Invalid"
        raise TypeError(f"Unexpected type for column {column}: {type(value)}")
    if "is_saccade_cued" in column:
        if isinstance(value, (bool, np.bool_)):
            return "Cued" if value else "Uncued"
        raise TypeError(f"Unexpected type for column {column}: {type(value)}")
    if "is_saccade_to_target" in column:
        if isinstance(value, (bool, np.bool_)):
            return "To Target" if value else "To Distractor"
        raise TypeError(f"Unexpected type for column {column}: {type(value)}")
    if "cue_size" in column:
        return et.CueSizeTypeEnum(value).name.capitalize()
    if hasattr(value, "name"):
        return value.name.capitalize()
    raise NotImplementedError(f"Unexpected columns and value combination: {(column, value)}")

### Load and Clean Data

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
CONDITION_SIZE_THRESHOLD = 10   # minimum number of trials per (experiment, subject, is_early) condition
ALLOW_TARGET_REPEATS = False    # exclude trials where the target location is repeated from the previous trial

DATA = load_and_prepare_experiments(
    DATA_DIR,
    min_condition_size=CONDITION_SIZE_THRESHOLD,
    allow_target_repeats=ALLOW_TARGET_REPEATS,
    verbose=True,
)

# add new columns per request by Dominique:
DATA = DATA.assign(
    is_saccade_cued=lambda df: df["saccade_cue"] != 0,
    is_saccade_to_target=lambda df: df["saccade_distractor_type"] == "TARGET",
)

print()
print(f"Data Features: {DATA.columns.tolist()}")

### Experiment 1
#### (A) Main Effect: Cue Validity

In [ ]:
COLORS = {
    (True, True): "lightblue",
    (True, False): "lightcoral",
    (False, True): "darkblue",
    (False, False): "darkred",
}

plot_by_condition(
    "exp_1",
    condition_columns=["is_early_saccade", "is_valid_cue"],
    color_map=COLORS,
    late_only=False,
)

#### (B) Main Effect: Search Difficulty

In [ ]:
COLORS = {  # (is_early, difficulty) -> color
    (True, et.SearchDifficultyTypeEnum.EASY): "lightgreen",
    (True, et.SearchDifficultyTypeEnum.MIXED): "gold",
    (True, et.SearchDifficultyTypeEnum.DIFFICULT): "lightcoral",
    (False, et.SearchDifficultyTypeEnum.EASY): "darkgreen",
    (False, et.SearchDifficultyTypeEnum.MIXED): "darkorange",
    (False, et.SearchDifficultyTypeEnum.DIFFICULT): "darkred",
}

plot_by_condition(
    "exp_1",
    condition_columns=["is_early_saccade", "search_difficulty"],
    color_map=COLORS,
    late_only=False,
)

#### (C) Interaction Effect: Search Difficulty × Cue Validity (Late Saccades Only)

In [ ]:
COLORS = {  # (is_valid, difficulty) -> color
    (et.SearchDifficultyTypeEnum.EASY, True): "lightgreen",
    (et.SearchDifficultyTypeEnum.MIXED, True): "gold",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True): "lightcoral",
    (et.SearchDifficultyTypeEnum.EASY, False): "darkgreen",
    (et.SearchDifficultyTypeEnum.MIXED, False): "darkorange",
    (et.SearchDifficultyTypeEnum.DIFFICULT, False): "darkred",
}

plot_by_condition(
    "exp_1",
    condition_columns=["search_difficulty", "is_valid_cue"],
    color_map=COLORS,
    late_only=True,
)

#### (D) Interaction Effect (Late Saccades Only): Search Difficulty × Is Saccade Cued × Is Saccade to Target

In [ ]:
COLORS = {  # (trial difficulty, is saccade cued, is saccade to target) -> color
    (et.SearchDifficultyTypeEnum.EASY, False, False): "#41ab5d",
    (et.SearchDifficultyTypeEnum.EASY, False, True): "#00441b",
    (et.SearchDifficultyTypeEnum.EASY, True, False): "#bdbdbd",
    (et.SearchDifficultyTypeEnum.EASY, True, True): "#525252",

    (et.SearchDifficultyTypeEnum.MIXED, False, False): "#efedf5",
    (et.SearchDifficultyTypeEnum.MIXED, False, True): "#bcbddc",
    (et.SearchDifficultyTypeEnum.MIXED, True, False): "#6a51a3",
    (et.SearchDifficultyTypeEnum.MIXED, True, True): "#4a1486",

    (et.SearchDifficultyTypeEnum.DIFFICULT, False, False): "#9e9ac8",
    (et.SearchDifficultyTypeEnum.DIFFICULT, False, True): "#3f007d",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True, False): "#fb6a4a",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True, True): "#67000d",
}

# exclude "MIXED" trials for visibility
DATA = DATA.loc[DATA["search_difficulty"] != et.SearchDifficultyTypeEnum.MIXED]

plot_by_condition(
    "exp_1",
    condition_columns=["search_difficulty", "is_saccade_cued", "is_saccade_to_target"],
    color_map=COLORS,
    late_only=True,
)

### Experiment 2
#### (A) Main Effect: Cue Validity

In [ ]:
COLORS = {
    (True, True): "lightblue",
    (True, False): "lightcoral",
    (False, True): "darkblue",
    (False, False): "darkred",
}

plot_by_condition(
    "exp_2",
    condition_columns=["is_early_saccade", "is_valid_cue"],
    color_map=COLORS,
    late_only=False,
)

#### (B) Main Effect: Cue Size

In [ ]:
COLORS = {  # (is_early, cue_size) -> color
    (True, et.CueSizeTypeEnum.SMALL): "lightgreen",
    (True, et.CueSizeTypeEnum.LARGE): "gold",
    (False, et.CueSizeTypeEnum.SMALL): "darkgreen",
    (False, et.CueSizeTypeEnum.LARGE): "darkorange",
}

plot_by_condition(
    "exp_2",
    condition_columns=["is_early_saccade", "cue_size"],
    color_map=COLORS,
    late_only=False,
)

#### (C) Main Effect: Search Difficulty

In [ ]:
COLORS = {  # (is_early, difficulty) -> color
    (True, et.SearchDifficultyTypeEnum.EASY): "lightgreen",
    (True, et.SearchDifficultyTypeEnum.MIXED): "gold",
    (True, et.SearchDifficultyTypeEnum.DIFFICULT): "lightcoral",
    (False, et.SearchDifficultyTypeEnum.EASY): "darkgreen",
    (False, et.SearchDifficultyTypeEnum.MIXED): "darkorange",
    (False, et.SearchDifficultyTypeEnum.DIFFICULT): "darkred",
}

plot_by_condition(
    "exp_2",
    condition_columns=["is_early_saccade", "search_difficulty"],
    color_map=COLORS,
    late_only=False,
)

#### (D) Interaction Effect: Cue Size × Cue Validity (Late Saccades Only)

In [ ]:
COLORS = {  # (is_valid, difficulty) -> color
    (et.CueSizeTypeEnum.SMALL, True): "lightgreen",
    (et.CueSizeTypeEnum.LARGE, True): "gold",
    (et.CueSizeTypeEnum.SMALL, False): "darkgreen",
    (et.CueSizeTypeEnum.LARGE, False): "darkorange",
}

plot_by_condition(
    "exp_2",
    condition_columns=["cue_size", "is_valid_cue"],
    color_map=COLORS,
    late_only=True,
)

#### (E) Interaction Effect: Search Difficulty × Cue Validity (Late Saccades Only)

In [ ]:
COLORS = {  # (is_valid, difficulty) -> color
    (et.SearchDifficultyTypeEnum.EASY, True): "lightgreen",
    (et.SearchDifficultyTypeEnum.MIXED, True): "gold",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True): "lightcoral",
    (et.SearchDifficultyTypeEnum.EASY, False): "darkgreen",
    (et.SearchDifficultyTypeEnum.MIXED, False): "darkorange",
    (et.SearchDifficultyTypeEnum.DIFFICULT, False): "darkred",
}

plot_by_condition(
    "exp_2",
    condition_columns=["search_difficulty", "is_valid_cue"],
    color_map=COLORS,
    late_only=True,
)

#### (D) Interaction Effect (Late Saccades Only): Search Difficulty × Is Saccade Cued × Is Saccade to Target

In [ ]:
COLORS = {  # (trial difficulty, is saccade cued, is saccade to target) -> color
    (et.SearchDifficultyTypeEnum.EASY, False, False): "#41ab5d",
    (et.SearchDifficultyTypeEnum.EASY, False, True): "#00441b",
    (et.SearchDifficultyTypeEnum.EASY, True, False): "#bdbdbd",
    (et.SearchDifficultyTypeEnum.EASY, True, True): "#525252",

    (et.SearchDifficultyTypeEnum.MIXED, False, False): "#efedf5",
    (et.SearchDifficultyTypeEnum.MIXED, False, True): "#bcbddc",
    (et.SearchDifficultyTypeEnum.MIXED, True, False): "#6a51a3",
    (et.SearchDifficultyTypeEnum.MIXED, True, True): "#4a1486",

    (et.SearchDifficultyTypeEnum.DIFFICULT, False, False): "#9e9ac8",
    (et.SearchDifficultyTypeEnum.DIFFICULT, False, True): "#3f007d",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True, False): "#fb6a4a",
    (et.SearchDifficultyTypeEnum.DIFFICULT, True, True): "#67000d",
}

plot_by_condition(
    "exp_2",
    condition_columns=["search_difficulty", "is_saccade_cued", "is_saccade_to_target"],
    color_map=COLORS,
    late_only=True,
)